In [1]:
# Lets create the LCEL RAG vanila pipe line using Openai API key 

%pip install langchain  langchain-openai  langchain-huggingface  langchain-qdrant qdrant-client

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os 
from dotenv import load_dotenv

# load env variable form dotenv
load_dotenv()

# get api key 
api_key = os.getenv("OPENAI_API_KEY")

# Verify the key is set (show only first 8 chars)
if api_key:
    print(f"API key loaded: {api_key[:8]}...")
else:
    print("❌ ERROR: API key not found!")
    print("Make sure you have a .env file with OPENAI_API_KEY")

API key loaded: sk-proj-...


In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


# component 1 : A prompt template with one variabel {topic}

prompt = ChatPromptTemplate.from_template(
    "Tell me one intersting fact about {topic} in 2 sentences."
)

# component 2 : The LLM (ChatGPT)
# temperature=0 means deteminsitic (same answer every time) great for facts 
# gpt-4o-mini is tast and cheap perfect for learning model 
llm = ChatOpenAI(model="gpt-4o-mini" , temperature=0)


# component 3: outpur parser extracts just the text string from LLM response 
parser = StrOutputParser()


# now Connect them with pip sign | (its the LCEL pipe operator)
# data flows left to right: {topic}-> prompt -> llm -> parser-> string
simple_chain = prompt | llm | parser



# .invoke() runs the chain with the given input 
result = simple_chain.invoke({"topic" : " NLP "})
print(result)

c:\Users\Dell\.conda\envs\hts_rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Natural Language Processing (NLP) has advanced significantly due to the development of deep learning techniques, enabling machines to understand and generate human language with remarkable accuracy. One interesting fact is that models like OpenAI's GPT-3, which has 175 billion parameters, can generate coherent and contextually relevant text, making them capable of tasks ranging from creative writing to coding assistance.


In [14]:
from qdrant_client import qdrant_client
import os 
from dotenv import load_dotenv

# load env variable form dotenv
load_dotenv()

# get api key and url of qdrant
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
QDRANT_URL= os.getenv("QDRANT_URL")


# Verify the key is set (show only first 8 chars)
if QDRANT_API_KEY:
    print(f"QDRANT API KEY key loaded: {QDRANT_API_KEY[:8]}...")
else:
    print("❌ ERROR: API key not found!")
    print("Make sure you have a .env file with QDRANT_API_KEY")

# Verify the QDRANT URL key is set (show only first 8 chars)
if QDRANT_URL:
    print(f"QDRANT URL key loaded: {QDRANT_URL}")
else:
    print("❌ ERROR: QDRANT URL key not found!")
    print("Make sure you have a .env file with QDRANT URL Key")


QDRANT API KEY key loaded: eyJhbGci...
QDRANT URL key loaded: https://04442bd4-e9e8-4779-93bd-3dfddabc7b4b.us-west-2-0.aws.cloud.qdrant.io:6333


In [6]:
%pip install -qU langchain-qdrant 

Note: you may need to restart the kernel to use updated packages.


In [15]:
# now building the search base knowledge base from lec 5 to 8 

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore

# Connect to Qdrant Cloud

# step 1 load document 
loader = TextLoader("data/nlp_article.txt", encoding="utf-8")
documents = loader.load()
print(f"The length of docuemnt is {len(documents)}")

# step 2 split the document 

spliter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 200,
)
chunks = spliter.split_documents(documents)
print(f"Total Number of Chunks {len(chunks)}")
print(f" \n Step 2 , split into  {chunks[:10]} chnunks" )

# step 3 & 4 Generate embedding and store in Qdrant store 
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = QdrantVectorStore.from_documents(
    documents = chunks,
    embedding= embeddings,
    url = QDRANT_URL,
    api_key = QDRANT_API_KEY,
    collection_name = "NLP_RAG_Pipline",
)

print(f"Step 3&4 - Embedded and stored {len(chunks)} chunks in Qdrant Cloud")
print(f"         Visit your dashboard to see the 'nlp_course' collection!")
print(f"\nKnowledge base ready!")

The length of docuemnt is 1
Total Number of Chunks 21
 
 Step 2 , split into  [Document(metadata={'source': 'data/nlp_article.txt'}, page_content='Natural Language Processing: A Comprehensive Guide\n\nChapter 1: What is Natural Language Processing?\n\nNatural Language Processing, commonly known as NLP, is a branch of artificial intelligence that focuses on the interaction between computers and humans through natural language. The ultimate objective of NLP is to read, decipher, understand, and make sense of human language in a manner that is both valuable and meaningful.'), Document(metadata={'source': 'data/nlp_article.txt'}, page_content="NLP combines computational linguistics — rule-based modeling of human language — with statistical, machine learning, and deep learning models. Together, these technologies enable computers to process human language in the form of text or voice data and to understand its full meaning, complete with the speaker's or writer's intent and sentiment."), Do

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2271.15it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step 3&4 - Embedded and stored 21 chunks in Qdrant Cloud
         Visit your dashboard to see the 'nlp_course' collection!

Knowledge base ready!


In [ ]:

# previous cell code updated wiht condition 
from qdrant_client import QdrantClient

# init client
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
collection_name = "NLP_RAG_Pipline"

# check if collection exists
collections = [c.name for c in client.get_collections().collections]

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

if collection_name in collections:
    print("Collection already exists , loading...")
    vector_store = QdrantVectorStore(
        client=client,
        collection_name=collection_name,
        embedding=embeddings,
    )
else:
    print("Creating new collection ")
    vector_store = QdrantVectorStore.from_documents(
        documents=chunks,
        embedding=embeddings,
        url=QDRANT_URL,
        api_key=QDRANT_API_KEY,
        collection_name=collection_name,
    )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2249.51it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Collection already exists ✅, loading...


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY:
    print(f"OPENAI API key loaded: {OPENAI_API_KEY[:8]}...")
else:
    print("❌ ERROR: OPENAI_API_KEY not found!")
    print("Make sure you have a .env file with OPENAI_API_KEY")

OPENAI API key loaded: sk-proj-...


In [9]:
from qdrant_client import QdrantClient
from dotenv import load_dotenv
import os

load_dotenv()

url = os.getenv("QDRANT_URL")
key = os.getenv("QDRANT_API_KEY")

if not url or not key:
    raise ValueError("Missing Qdrant credentials")

client = QdrantClient(url=url, api_key=key)
client.get_collections()

CollectionsResponse(collections=[CollectionDescription(name='redundancy_demo'), CollectionDescription(name='NLP_RAG_Pipline')])

In [11]:
# ============================================================
# BUILD THE KNOWLEDGE BASE — OpenAI Embeddings Version
# ============================================================
# EMBEDDING METHOD: OpenAI Embeddings (PAID, requires API key)
#
# This cell uses OpenAI embeddings (requires API key and costs money)
# For FREE local embeddings, use the previous cell instead
#
# Same pipeline, but with OpenAI's text-embedding-3-small model
# ============================================================

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore

print("=" * 70)
print("KNOWLEDGE BASE SETUP - OpenAI Embeddings Version")
print("=" * 70)

# --- Step 1: LOAD (Lecture 5) ---
loader = TextLoader("data/nlp_article.txt", encoding="utf-8")
documents = loader.load()
print(f"[SUCCESS] Step 1 - Loaded: {len(documents)} document(s)")

# --- Step 2: SPLIT (Lecture 6) ---
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,     # ~500 characters per chunk
    chunk_overlap=50,   # 50-char overlap so context isn't lost at boundaries
)
chunks = splitter.split_documents(documents)
print(f"[SUCCESS] Step 2 - Split into: {len(chunks)} chunks")


# step 3 and 4 , openai embedding model initilized 

print(" Open AI embedding modle initilized 1536 dimensions")
embeddings_opanai = OpenAIEmbeddings(model="text-embedding-3-small")
print(f"[SUCCESS] OpenAI embedding model initialized (1536 dimensions)")

# now user same as previous cell and store in qdrant store with diffrent collection name 

vector_store_with_openai = QdrantVectorStore.from_documents(
    documents= chunks,
    embedding= embeddings_opanai,
    url = url,
    api_key = key,
    collection_name = "NLP_RAG_OpenAI",
)

print(f"[SUCCESS] Embedded and stored {len(chunks)} chunks in Qdrant Cloud")
print(f"  Collection name: nlp_course_openai")
print(f"  Embedding dimensions: 1536")
print(f"  Model: text-embedding-3-small")
print(f"  Estimated cost: ~$0.02 per 1M tokens")
print(f"\n" + "=" * 70)
print("KNOWLEDGE BASE READY - OpenAI Version")
print("=" * 70)
print(f"\n[INFO] Comparison:")
print(f"  - FREE (SentenceTransformer): 384 dims, runs locally, $0 cost")
print(f"  - PAID (OpenAI): 1536 dims, API-based, ~$0.02/1M tokens")
print(f"\n[INFO] Visit your Qdrant dashboard to see both collections!")
print(f"  - nlp_course (free)")
print(f"  - nlp_course_openai (paid)")

KNOWLEDGE BASE SETUP - OpenAI Embeddings Version
[SUCCESS] Step 1 - Loaded: 1 document(s)
[SUCCESS] Step 2 - Split into: 45 chunks
 Open AI embedding modle initilized 1536 dimensions
[SUCCESS] OpenAI embedding model initialized (1536 dimensions)
[SUCCESS] Embedded and stored 45 chunks in Qdrant Cloud
  Collection name: nlp_course_openai
  Embedding dimensions: 1536
  Model: text-embedding-3-small
  Estimated cost: ~$0.02 per 1M tokens

KNOWLEDGE BASE READY - OpenAI Version

[INFO] Comparison:
  - FREE (SentenceTransformer): 384 dims, runs locally, $0 cost
  - PAID (OpenAI): 1536 dims, API-based, ~$0.02/1M tokens

[INFO] Visit your Qdrant dashboard to see both collections!
  - nlp_course (free)
  - nlp_course_openai (paid)


In [12]:
# ============================================================
# BUILD THE KNOWLEDGE BASE — OpenAI Embeddings Version
# ============================================================
# EMBEDDING METHOD: OpenAI Embeddings (PAID, requires API key)
#
# This cell uses OpenAI embeddings (requires API key and costs money)
# For FREE local embeddings, use the previous cell instead
#
# Same pipeline, but with OpenAI's text-embedding-3-small model
# ============================================================

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore

print("=" * 70)
print("KNOWLEDGE BASE SETUP - OpenAI Embeddings Version")
print("=" * 70)

# --- Step 1: LOAD (Lecture 5) ---
loader = TextLoader("data/nlp_article.txt", encoding="utf-8")
documents = loader.load()
print(f"[SUCCESS] Step 1 - Loaded: {len(documents)} document(s)")

# --- Step 2: SPLIT (Lecture 6) ---
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,     # ~500 characters per chunk
    chunk_overlap=50,   # 50-char overlap so context isn't lost at boundaries
)
chunks = splitter.split_documents(documents)
print(f"[SUCCESS] Step 2 - Split into: {len(chunks)} chunks")


# step 3 and 4 , openai embedding model initilized 

print(" Open AI embedding modle initilized 1536 dimensions")
embeddings_opanai = OpenAIEmbeddings(model="text-embedding-3-small")
print(f"[SUCCESS] OpenAI embedding model initialized (1536 dimensions)")

# now use same as previous cell and store in qdrant store with diffrent collection name 

# Check if collection exists to avoid 403 errors
from qdrant_client import QdrantClient

client_openai = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
collection_name_openai = "NLP_RAG_OpenAI"

# Get list of existing collections
collections_openai = [c.name for c in client_openai.get_collections().collections]


if collection_name_openai in collections_openai:
    print("Collection already exists, deleting and recreating to ensure correct embeddings...")
    client_openai.delete_collection(collection_name=collection_name_openai)
    print("Creating new collection...")
    vector_store_with_openai = QdrantVectorStore.from_documents(
        documents=chunks,
        embedding=embeddings_opanai,
        url=url,
        api_key=key,
        collection_name=collection_name_openai,
    )
else:
    print("Creating new collection...")
    vector_store_with_openai = QdrantVectorStore.from_documents(
        documents=chunks,
        embedding=embeddings_opanai,
        url=QDRANT_URL,
        api_key=QDRANT_API_KEY,
        collection_name=collection_name_openai,
    )

print(f"[SUCCESS] Embedded and stored {len(chunks)} chunks in Qdrant Cloud")
print(f"  Collection name: nlp_course_openai")
print(f"  Embedding dimensions: 1536")
print(f"  Model: text-embedding-3-small")
print(f"  Estimated cost: ~$0.02 per 1M tokens")
print(f"\n" + "=" * 70)
print("KNOWLEDGE BASE READY - OpenAI Version")
print("=" * 70)
print(f"\n[INFO] Comparison:")
print(f"  - FREE (SentenceTransformer): 384 dims, runs locally, $0 cost")
print(f"  - PAID (OpenAI): 1536 dims, API-based, ~$0.02/1M tokens")
print(f"\n[INFO] Visit your Qdrant dashboard to see both collections!")
print(f"  - nlp_course (free)")
print(f"  - nlp_course_openai (paid)")

KNOWLEDGE BASE SETUP - OpenAI Embeddings Version
[SUCCESS] Step 1 - Loaded: 1 document(s)
[SUCCESS] Step 2 - Split into: 21 chunks
 Open AI embedding modle initilized 1536 dimensions
[SUCCESS] OpenAI embedding model initialized (1536 dimensions)
Collection already exists, deleting and recreating to ensure correct embeddings...
Creating new collection...
[SUCCESS] Embedded and stored 21 chunks in Qdrant Cloud
  Collection name: nlp_course_openai
  Embedding dimensions: 1536
  Model: text-embedding-3-small
  Estimated cost: ~$0.02 per 1M tokens

KNOWLEDGE BASE READY - OpenAI Version

[INFO] Comparison:
  - FREE (SentenceTransformer): 384 dims, runs locally, $0 cost
  - PAID (OpenAI): 1536 dims, API-based, ~$0.02/1M tokens

[INFO] Visit your Qdrant dashboard to see both collections!
  - nlp_course (free)
  - nlp_course_openai (paid)


In [19]:
# now build the prompt that run on runtime having 
# having two factores , 1 is context that have the retrived chunks from vector db 
# and 2nd is the query from user , 
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_template(
"""
You are a helpful and concise teaching assistant for an NLP course.

INSTRUCTIONS:
- Use ONLY the provided context to answer.
- If the answer is not in the context, say: "I don't have enough information."
- Be clear, short, and structured like ChatGPT.
- Prefer bullet points where helpful.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER FORMAT:
- Direct answer first (1–2 lines)
- Then bullet points (if needed)
- Keep it simple and exam-friendly
"""
)

# Let's inspect the template
print("Prompt Template Variables:")
# .input_variables shows what variables the template expects
print(f"  Required inputs: {rag_prompt.input_variables}")

# sample of rag_prompt 
sample = rag_prompt.format(
    context="NLP is a branch of AI that focuses on human language...",
    question="What is NLP"
)

print(f"\nSample formatted prompt:")
print(f"{'=' * 50}")
# [:300] shows first 300 chars of the formatted prompt
# Remove [:300] to see the entire prompt
print(sample[:300])

Prompt Template Variables:
  Required inputs: ['context', 'question']

Sample formatted prompt:
Human: 
You are a helpful and concise teaching assistant for an NLP course.

INSTRUCTIONS:
- Use ONLY the provided context to answer.
- If the answer is not in the context, say: "I don't have enough information."
- Be clear, short, and structured like ChatGPT.
- Prefer bullet points where helpful.




In [24]:
# ============================================================
# ASSEMBLING THE RAG CHAIN
# ============================================================

from langchain_openai  import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# component 1 retriver 
retriver = vector_store.as_retriever(search_kwargs= {"k" : 5 } )

# --- Helper: FORMAT DOCS ---
# The retriever returns Document objects, but the prompt needs a string
# This function joins all retrieved chunks into one text block
def format_docs(docs):
    """Join retrieved document chunks into a single context string."""
    # "\n\n" puts a blank line between each chunk for readability
    # doc.page_content gets the text from each Document object
    # format_docs = results ko ek paragraph bana deta hai
    return "\n\n".join(doc.page_content for doc in docs)

# --- Component 2: PROMPT (created in Section 5 above) ---
# rag_prompt is our ChatPromptTemplate with {context} and {question}

# --- Component 3: LLM ---
# temperature=0 means deterministic — same question always gets same answer
# This is important for factual RAG (no creative variation)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# --- Component 4: OUTPUT PARSER ---
parser = StrOutputParser()



# ============================================================
# THE CHAIN — connect everything with |
# ============================================================


# The Chain --- connect every thing

rag_chain= (
    {
        # contexxt' : retriver finds docs, then formate_doc join them 
        "context" : retriver | format_docs, 
        "question" : RunnablePassthrough(),
    }
    | rag_prompt     # fills {context} and {question} into the template
    | llm            # sends the formatted prompt to chatgpt
    | parser         # Extracts just the text string from the response 
)

print("RAG chain assembled!")
print(f"\nChain components:")
print(f"  1. Retriever: Qdrant vector store (top 5 chunks)")
print(f"  2. Prompt: RAG template with context + question")
print(f"  3. LLM: gpt-4o-mini (temperature=0)")
print(f"  4. Parser: StrOutputParser (plain text output)")


RAG chain assembled!

Chain components:
  1. Retriever: Qdrant vector store (top 5 chunks)
  2. Prompt: RAG template with context + question
  3. LLM: gpt-4o-mini (temperature=0)
  4. Parser: StrOutputParser (plain text output)


In [25]:
# Time to test our RAG chain 
# lets ask our first question 

question="What is Natural Language Processing"

# .invoke() runs the entire chain:
# question -> retriever -> format_docs -> prompt -> LLM -> parser -> answer

answer = rag_chain.invoke(question)

print(f" Quesiont: {question}")
print(f"=" * 60 )
print(f"\n Answer: {answer}")



 Quesiont: What is Natural Language Processing

 Answer: Natural Language Processing (NLP) is a branch of artificial intelligence focused on the interaction between computers and humans through natural language, aiming to read, understand, and make sense of human language.

- Combines computational linguistics with statistical, machine learning, and deep learning models.
- Processes text or voice data to understand meaning, intent, and sentiment.


In [30]:
# test with diffrent question 

question="what is hope to skill "

retriver_doc=retriver.invoke(question)


print(f"Lenght of retriver document from vector store {len(retriver_doc)}")

for i , doc in enumerate(retriver_doc):
    print(f"\n chunk {[i+1]}")
    # print first 50 charctero 
    print(f" {doc.page_content[:150]}")

# now run the full chain to seen anser 
print(f"=" * 50 )

print("Full chain Answer")

answer=rag_chain.invoke(question)
print(answer)

Lenght of retriver document from vector store 5

 chunk [1]
 GPT (Generative Pre-trained Transformer) models, developed by OpenAI, take a different approach. They are trained to predict the next word in a sequen

 chunk [2]
 Retrieval-Augmented Generation (RAG) combines retrieval and generation to produce accurate, grounded responses. Instead of relying solely on the LLM's

 chunk [3]
 The transformer architecture, introduced in the landmark paper "Attention Is All You Need" by Vaswani et al. in 2017, revolutionized NLP. Unlike previ

 chunk [4]
 Evaluation metrics matter. Different NLP tasks require different evaluation metrics. For classification tasks, use accuracy, precision, recall, and F1

 chunk [5]
 Monitor and maintain. NLP models can degrade over time as language evolves and data distributions shift. Implement monitoring to track model performan
Full chain Answer
I don't have enough information.


In [38]:
# lets try diffrent question to test our rag chain 

questions = [
    "What are the main NLP tasks?",
    "Who created the transformer architecture and when?",
    "What is RAG and how does it work?",
]

for i , question in enumerate(questions):
    print(f" Question : {[i+1]} {question}")
    print(f"="*50)
    print(f"\n")

    answer=rag_chain.invoke(question)

    print(answer)
    print(f"\n")

 Question : [1] What are the main NLP tasks?


The main NLP tasks include text classification, sentiment analysis, and topic categorization. 

- Text Classification: Assigning predefined categories to text documents.
- Sentiment Analysis: Determining the sentiment expressed in text, such as positive or negative.
- Topic Categorization: Classifying text into specific topics or themes.


 Question : [2] Who created the transformer architecture and when?


- The transformer architecture was created by Vaswani et al. in 2017.

- Key points:
  - Introduced in the paper "Attention Is All You Need."
  - Revolutionized NLP by enabling parallel processing of text.


 Question : [3] What is RAG and how does it work?


**RAG (Retrieval-Augmented Generation)** is a method that combines document retrieval and language generation to produce accurate and grounded responses. It retrieves relevant documents from a knowledge base and uses them as context for generating answers.

- Reduces hallucination 

In [46]:
# use stremming to see word prints word by word 

question="Explain how BERT works in simple terms."

print(f"Question {question}")
print(f"-"*50)
print(f"Streaming Answer :" )

streaming_answers= rag_chain.stream(question)

for i , ans in enumerate(streaming_answers):
    print(ans , end="" , flush=True)
    # Add a newline at the end so the next print starts on a new line
print("\n")
print(f"{'=' * 60}")
print("The answer streamed token by token — much better UX!")


Question Explain how BERT works in simple terms.
--------------------------------------------------
Streaming Answer :
BERT works by reading text in both directions (left and right) to understand the context of each word, which helps it grasp the meaning of sentences better.

- It is a pre-trained model that learns from large amounts of text data.
- After pre-training, BERT can be fine-tuned for specific tasks like text classification.
- This bidirectional reading allows BERT to capture nuances and relationships in language effectively.

The answer streamed token by token — much better UX!


In [47]:
# Test with 5 different questions!
import time

def ask_rag(question , chain=rag_chain, ret=retriver , show_sources=True):

    print(f"\nQuestion: {question}")
    print(f"{'=' * 65}")

    # time for eniter RAG Pipline 
    time_start=time.time()

    # run the chain to get answer 
    answer = rag_chain.invoke(question)

    # Calculate how long it took
    # (time.time() - start_time) gives seconds, * 1000 converts to ms
    elapsed_ms = (time.time() - time_start) * 1000

    print(f"\nAnswer:\n{answer}")
    # :.0f formats the number with zero decimal places
    print(f"\n--- Latency: {elapsed_ms:.0f} ms ---")

        # Optionally show which chunks were retrieved
    if show_sources:
        retrieved_docs = ret.invoke(question)
        print(f"\n--- Sources ({len(retrieved_docs)} chunks retrieved) ---")
        # This loop shows a preview of each source chunk
        for i, doc in enumerate(retrieved_docs):
            # [:100] shows first 100 chars; remove to see full chunk
            print(f"  [{i + 1}] {doc.page_content[:100]}...")

    return answer
    
questions=[
    "What is Natural Language Processing?",
    "What are the main NLP tasks described in the article?",
    "How do transformers work?",
    "What is LangChain used for?",
    "What are best practices for NLP projects?",
]

# This loop asks each question using our ask_rag function
for question in questions:
    ask_rag(question)
    print()


Question: What is Natural Language Processing?

Answer:
Natural Language Processing (NLP) is a branch of artificial intelligence focused on the interaction between computers and humans through natural language, aiming to read, understand, and make sense of human language.

- Combines computational linguistics with statistical, machine learning, and deep learning models.
- Processes text or voice data to understand meaning, intent, and sentiment.

--- Latency: 4733 ms ---

--- Sources (5 chunks retrieved) ---
  [1] Natural Language Processing: A Comprehensive Guide

Chapter 1: What is Natural Language Processing?
...
  [2] NLP combines computational linguistics — rule-based modeling of human language — with statistical, m...
  [3] Chapter 2: Core NLP Tasks

Text Classification is one of the most fundamental tasks in NLP. It invol...
  [4] The transformer architecture, introduced in the landmark paper "Attention Is All You Need" by Vaswan...
  [5] The history of NLP dates back to the 19